# Camada Silver — `alunos`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Limpeza de Dados Textuais, Correção de Mojibake, Padronização de Formatos (`sexo`, `renda_estimada`, `data_nascimento`), Validação de Regras de Negócio (`idade` [15, 90]), Imputação por Mediana/Moda, Detecção de Outliers (IQR e Z-Score), Binning (`faixa_etaria`, `faixa_renda`), Remoção de Duplicatas e Anonimização LGPD (`nome`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("alunos", BRONZE_DIR)
rows_in = len(df)

# 1. Padronização de chaves e tipos (id_aluno -> Int64, data_nascimento -> datetime)
df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
df["data_nascimento"] = parse_date_col(df["data_nascimento"])

# 2. LGPD: Anonimização de nome (pseudônimo SHA-256) e remoção do dado pessoal em claro
df["nome_anonimizado"] = anonymize_name_lgpd(df["nome"], df["id_aluno"])
df = df.drop(columns=["nome"])

# 3. Limpeza de dados textuais e correção de mojibake em cidade
df["cidade"], mojibake_cidade = fix_mojibake_text(df["cidade"])
df["cidade"] = df["cidade"].replace("", "Não Informada").str.title()
cidade_imputadas = int((df["cidade"] == "Não Informada").sum())

# 4. Correção de inconsistências categóricas (sexo)
sexo_map = {
    "f": "F",
    "fem": "F",
    "feminino": "F",
    "m": "M",
    "masculino": "M",
    "x": "Outro",
    "n/a": "Não Informado",
    "": "Não Informado",
}
df["sexo"] = df["sexo"].astype(str).str.strip().str.lower().map(sexo_map).fillna("Não Informado").astype("category")

# 5. Validação de regra de negócio em idade ([15, 90]) + Imputação por Mediana + Binning
idade_num = pd.to_numeric(df["idade"], errors="coerce")
idade_invalida_mask = idade_num.isna() | (idade_num < 15) | (idade_num > 90)
idade_mediana = float(idade_num.loc[~idade_invalida_mask].median())
df["flag_idade_imputada"] = idade_invalida_mask
df["idade"] = idade_num.where(~idade_invalida_mask, idade_mediana).round().astype("Int64")
df["faixa_etaria"] = pd.cut(
    df["idade"].astype(float),
    bins=[0, 20, 25, 35, 50, 120],
    labels=["<=20", "21-25", "26-35", "36-50", ">50"],
).astype("category")

# 6. Padronização decimal de renda_estimada + Imputação por Mediana + Outliers (IQR / Z-Score) + Binning
renda_num = parse_br_float(df["renda_estimada"])
renda_invalida_mask = renda_num.isna() | (renda_num < 0)
renda_mediana = float(renda_num.loc[~renda_invalida_mask].median())
df["flag_renda_imputada"] = renda_invalida_mask
df["renda_estimada"] = renda_num.where(~renda_invalida_mask, renda_mediana)
df["flag_renda_outlier_iqr"], iqr_info = detect_outliers_iqr(df["renda_estimada"])
df["flag_renda_outlier_zscore"], z_count = detect_outliers_zscore(df["renda_estimada"])
df["faixa_renda"] = pd.qcut(
    df["renda_estimada"],
    q=4,
    labels=["Q1 (Baixa)", "Q2 (Média-Baixa)", "Q3 (Média-Alta)", "Q4 (Alta)"],
).astype("category")

# 7. Remoção de duplicatas
dups_removed = int(df.duplicated(subset=["id_aluno"]).sum())
df = df.drop_duplicates(subset=["id_aluno"]).reset_index(drop=True)

log = {
    "dataset": "alunos",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "lgpd_nome_anonymized": True,
    "mojibake_fixed_cidade": mojibake_cidade,
    "cidade_imputada_nao_informada": cidade_imputadas,
    "idade_imputada_mediana": int(idade_invalida_mask.sum()),
    "idade_mediana_valor": idade_mediana,
    "renda_imputada_mediana": int(renda_invalida_mask.sum()),
    "renda_mediana_valor": round(renda_mediana, 2),
    "renda_outliers_iqr": iqr_info,
    "renda_outliers_zscore": z_count,
}
p_out, l_out = save_silver_outputs(df, log, "alunos", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)